In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import scipy.optimize as sco
from datetime import datetime

print("All libraries loaded successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

All libraries loaded successfully
Pandas version: 1.5.2
NumPy version: 1.24.1


In [3]:
tickers = [
    'SPY',  # US equities
    'EFA',  # International developed equities
    'EEM',  # Emerging markets
    'TLT',  # Long-term US treasuries
    'GLD',  # Gold
    'VNQ',  # Real estate (REITs)
    'XLE',  # Energy
    'XLF',  # Financials
    'XLV',  # Healthcare
    'DBC',  # Commodities
]

start = '2014-01-01'
end = datetime.today().strftime('%Y-%m-%d')

print(f"Downloading data from {start} to {end}...")
raw = yf.download(tickers, start=start, end=end, auto_adjust=True)
prices = raw['Close'].dropna()

print(f"\nShape: {prices.shape}")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
prices.head()

[*********************100%***********************]  10 of 10 completed


Shape: (3111, 10)
Date range: 2014-01-02 to 2026-05-15


Ticker,DBC,EEM,EFA,GLD,SPY,TLT,VNQ,XLE,XLF,XLV
Date,,,,,,,,,,
2014-01-02,21.381117,30.777197,46.040524,118.000000,148.580292,72.757469,39.707462,27.449310,14.067754,45.176399
2014-01-03,21.245844,30.723598,46.089428,119.290001,148.555862,72.757469,39.928848,27.348639,14.164814,45.282917
2014-01-06,21.271210,30.432590,46.033546,119.500000,148.125427,73.063705,40.131798,27.386414,14.177758,45.094448
2014-01-07,21.254299,30.562778,46.291992,118.820000,149.035187,73.248901,40.310120,27.594042,14.184228,45.569725
2014-01-08,21.051395,30.463221,46.278027,118.120003,149.067581,73.049484,40.113342,27.402130,14.229522,45.971268


In [4]:
log_returns = np.log(prices / prices.shift(1)).dropna()

ann_vol = log_returns.std() * np.sqrt(252)
ann_ret = log_returns.mean() * 252

summary = pd.DataFrame({
    'Ann. Return (%)': (ann_ret * 100).round(2),
    'Ann. Volatility (%)': (ann_vol * 100).round(2),
    'Sharpe (approx)': ((ann_ret - 0.05) / ann_vol).round(2)
})

print("Asset summary statistics:")
print(summary.sort_values('Sharpe (approx)', ascending=False).to_string())

Asset summary statistics:
        Ann. Return (%)  Ann. Volatility (%)  Sharpe (approx)
Ticker                                                       
SPY               13.00                17.28             0.46
GLD               10.23                15.84             0.33
XLV                9.45                16.59             0.27
XLF               10.45                21.24             0.26
VNQ                6.97                19.91             0.10
EFA                6.42                17.03             0.08
EEM                6.07                20.38             0.05
XLE                6.26                28.66             0.04
DBC                3.06                17.52            -0.11
TLT                1.13                14.60            -0.26


In [5]:
lookback = 252
skip = 21

momentum = prices.shift(skip) / prices.shift(lookback) - 1

print("Momentum signal (last 5 rows):")
print(momentum.tail().round(4).to_string())
print(f"\nSignal starts from: {momentum.dropna().index[0].date()}")

Momentum signal (last 5 rows):
Ticker         DBC     EEM     EFA     GLD     SPY     TLT     VNQ     XLE     XLF     XLV
Date                                                                                      
2026-05-11  0.4055  0.3903  0.2371  0.4350  0.2162  0.0364  0.0915  0.4402  0.0340  0.1178
2026-05-12  0.4174  0.3927  0.2386  0.4189  0.2297  0.0380  0.0884  0.4301  0.0521  0.1351
2026-05-13  0.4022  0.3893  0.2423  0.4926  0.2049  0.0533  0.0934  0.3655  0.0336  0.1142
2026-05-14  0.3832  0.3860  0.2367  0.4708  0.2064  0.0529  0.1083  0.3420  0.0371  0.1406
2026-05-15  0.4008  0.3811  0.2392  0.5012  0.2078  0.0533  0.1296  0.3700  0.0372  0.1588

Signal starts from: 2015-01-02


In [6]:
monthly_momentum = momentum.resample('M').last()
monthly_prices = prices.resample('M').last()
monthly_returns = monthly_prices.pct_change().shift(-1)

n_long = 3
n_short = 3

portfolio_returns = []
dates = []
long_assets_history = []
short_assets_history = []

for date, row in monthly_momentum.iterrows():
    if row.isna().any():
        continue
    
    ranked = row.sort_values(ascending=False)
    long_assets = ranked.index[:n_long].tolist()
    short_assets = ranked.index[-n_short:].tolist()
    
    if date in monthly_returns.index:
        next_returns = monthly_returns.loc[date]
        long_ret = next_returns[long_assets].mean()
        short_ret = next_returns[short_assets].mean()
        portfolio_ret = long_ret - short_ret
        
        portfolio_returns.append(portfolio_ret)
        dates.append(date)
        long_assets_history.append(', '.join(long_assets))
        short_assets_history.append(', '.join(short_assets))

results = pd.DataFrame({
    'return': portfolio_returns,
    'long': long_assets_history,
    'short': short_assets_history
}, index=dates)

print(f"Strategy runs from {results.index[0].date()} to {results.index[-1].date()}")
print(f"Total months: {len(results)}")
print(f"\nSample of positions taken:")
print(results[['long', 'short']].head(8).to_string())

Strategy runs from 2015-01-31 to 2026-05-31
Total months: 137

Sample of positions taken:
                     long          short
2015-01-31  VNQ, XLV, TLT  XLE, GLD, DBC
2015-02-28  VNQ, TLT, XLV  EFA, XLE, DBC
2015-03-31  XLV, VNQ, TLT  GLD, XLE, DBC
2015-04-30  XLV, TLT, VNQ  GLD, XLE, DBC
2015-05-31  XLV, TLT, VNQ  GLD, XLE, DBC
2015-06-30  XLV, VNQ, SPY  GLD, XLE, DBC
2015-07-31  XLV, XLF, SPY  GLD, XLE, DBC
2015-08-31  XLV, XLF, SPY  EEM, XLE, DBC


In [7]:
results_clean = results.dropna(subset=['return']).copy()

cum_returns = (1 + results_clean['return']).cumprod()

n_years = len(results_clean) / 12
total_return = cum_returns.iloc[-1] - 1
ann_return = (1 + total_return) ** (1 / n_years) - 1
ann_vol = results_clean['return'].std() * np.sqrt(12)
rf = 0.02
sharpe = (ann_return - rf) / ann_vol
rolling_max = cum_returns.cummax()
drawdown = (cum_returns - rolling_max) / rolling_max
max_drawdown = drawdown.min()
hit_rate = (results_clean['return'] > 0).mean()
downside_vol = results_clean['return'][results_clean['return'] < 0].std() * np.sqrt(12)
sortino = (ann_return - rf) / downside_vol

print("=" * 45)
print("   MOMENTUM STRATEGY — PERFORMANCE SUMMARY")
print("=" * 45)
print(f"  Period:            {results_clean.index[0].date()} to {results_clean.index[-1].date()}")
print(f"  Annualised Return: {ann_return*100:.2f}%")
print(f"  Annualised Vol:    {ann_vol*100:.2f}%")
print(f"  Sharpe Ratio:      {sharpe:.2f}")
print(f"  Sortino Ratio:     {sortino:.2f}")
print(f"  Max Drawdown:      {max_drawdown*100:.2f}%")
print(f"  Hit Rate:          {hit_rate*100:.1f}% of months profitable")
print(f"  Total Return:      {total_return*100:.2f}%")
print("=" * 45)

   MOMENTUM STRATEGY — PERFORMANCE SUMMARY
  Period:            2015-01-31 to 2026-04-30
  Annualised Return: 3.04%
  Annualised Vol:    17.32%
  Sharpe Ratio:      0.06
  Sortino Ratio:     0.10
  Max Drawdown:      -36.50%
  Hit Rate:          54.4% of months profitable
  Total Return:      40.44%


In [8]:
cost_per_trade = 0.001
positions_per_month = 6
monthly_cost = cost_per_trade * positions_per_month

results_clean['return_after_costs'] = results_clean['return'] - monthly_cost

cum_returns_costs = (1 + results_clean['return_after_costs']).cumprod()

total_return_costs = cum_returns_costs.iloc[-1] - 1
ann_return_costs = (1 + total_return_costs) ** (1 / n_years) - 1
sharpe_costs = (ann_return_costs - rf) / ann_vol

print("=" * 45)
print("   IMPACT OF TRANSACTION COSTS")
print("=" * 45)
print(f"  Monthly cost drag:     {monthly_cost*100:.2f}%")
print(f"  Annual cost drag:      {monthly_cost*12*100:.2f}%")
print()
print(f"  Gross Sharpe:          {sharpe:.2f}")
print(f"  Net Sharpe:            {sharpe_costs:.2f}")
print()
print(f"  Gross Annual Return:   {ann_return*100:.2f}%")
print(f"  Net Annual Return:     {ann_return_costs*100:.2f}%")
print()
print(f"  Gross Total Return:    {total_return*100:.2f}%")
print(f"  Net Total Return:      {total_return_costs*100:.2f}%")
print("=" * 45)

   IMPACT OF TRANSACTION COSTS
  Monthly cost drag:     0.60%
  Annual cost drag:      7.20%

  Gross Sharpe:          0.06
  Net Sharpe:            -0.35

  Gross Annual Return:   3.04%
  Net Annual Return:     -4.13%

  Gross Total Return:    40.44%
  Net Total Return:      -37.98%


In [9]:
threshold = 2
filtered_returns = []
filtered_dates = []
current_longs = None
current_shorts = None
trades_per_month = []

for date, row in monthly_momentum.iterrows():
    if row.isna().any():
        continue

    ranked = row.sort_values(ascending=False)
    new_longs = set(ranked.index[:n_long])
    new_shorts = set(ranked.index[-n_short:])

    if current_longs is None:
        current_longs = new_longs
        current_shorts = new_shorts
        trades = positions_per_month
    else:
        long_changes = len(new_longs - current_longs)
        short_changes = len(new_shorts - current_shorts)
        trades = long_changes + short_changes

        if long_changes > 0:
            current_longs = new_longs
        if short_changes > 0:
            current_shorts = new_shorts

    if date in monthly_returns.index:
        next_returns = monthly_returns.loc[date]
        if next_returns[list(current_longs)].isna().any():
            continue

        long_ret = next_returns[list(current_longs)].mean()
        short_ret = next_returns[list(current_shorts)].mean()
        port_ret = long_ret - short_ret
        actual_cost = trades * cost_per_trade
        net_ret = port_ret - actual_cost

        filtered_returns.append(net_ret)
        filtered_dates.append(date)
        trades_per_month.append(trades)

filtered = pd.DataFrame({
    'return': filtered_returns,
    'trades': trades_per_month
}, index=filtered_dates)

cum_filtered = (1 + filtered['return']).cumprod()
total_filtered = cum_filtered.iloc[-1] - 1
ann_filtered = (1 + total_filtered) ** (1 / n_years) - 1
vol_filtered = filtered['return'].std() * np.sqrt(12)
sharpe_filtered = (ann_filtered - rf) / vol_filtered
avg_trades = filtered['trades'].mean()

print("=" * 45)
print("   WITH TURNOVER FILTER")
print("=" * 45)
print(f"  Avg trades per month:  {avg_trades:.1f} (vs 6.0 naive)")
print(f"  Avg monthly cost:      {avg_trades * cost_per_trade * 100:.2f}% (vs {monthly_cost*100:.2f}%)")
print()
print(f"  Gross Sharpe:          {sharpe:.2f}")
print(f"  Naive net Sharpe:      {sharpe_costs:.2f}")
print(f"  Filtered net Sharpe:   {sharpe_filtered:.2f}")
print()
print(f"  Gross Annual Return:   {ann_return*100:.2f}%")
print(f"  Naive net Return:      {ann_return_costs*100:.2f}%")
print(f"  Filtered net Return:   {ann_filtered*100:.2f}%")
print("=" * 45)

   WITH TURNOVER FILTER
  Avg trades per month:  1.2 (vs 6.0 naive)
  Avg monthly cost:      0.12% (vs 0.60%)

  Gross Sharpe:          0.06
  Naive net Sharpe:      -0.35
  Filtered net Sharpe:   -0.02

  Gross Annual Return:   3.04%
  Naive net Return:      -4.13%
  Filtered net Return:   1.59%


In [10]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [3, 1]})

cum_filtered_plot = (1 + filtered['return']).cumprod()

ax1.plot(cum_returns.index, cum_returns.values,
         label=f'Gross (Sharpe: {sharpe:.2f})', 
         color='#2196F3', linewidth=1.5)
ax1.plot(cum_returns_costs.index, cum_returns_costs.values,
         label=f'Naive net (Sharpe: {sharpe_costs:.2f})', 
         color='#F44336', linewidth=1.5, linestyle='--')
ax1.plot(cum_filtered_plot.index, cum_filtered_plot.values,
         label=f'Turnover-filtered net (Sharpe: {sharpe_filtered:.2f})', 
         color='#4CAF50', linewidth=1.5, linestyle='-.')
ax1.axhline(y=1, color='gray', linestyle=':', linewidth=0.8)
ax1.set_ylabel('Portfolio Value ($1 invested)', fontsize=11)
ax1.set_title('12-1 Momentum Strategy — Gross vs Net vs Turnover-Filtered (2015–2026)', 
              fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

rolling_max_f = cum_filtered_plot.cummax()
drawdown_f = (cum_filtered_plot - rolling_max_f) / rolling_max_f
ax2.fill_between(drawdown_f.index, drawdown_f.values, 0,
                 color='#4CAF50', alpha=0.4, label='Filtered drawdown')
ax2.set_ylabel('Drawdown', fontsize=11)
ax2.set_xlabel('Date', fontsize=11)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('equity_curve_comparison.png', dpi=150, bbox_inches='tight')
print("Saved: equity_curve_comparison.png")

Saved: equity_curve_comparison.png
